In [1]:
import numpy as np
import pandas as pd

_orig_to_latex = pd.DataFrame.to_latex
def _to_latex(self, *args, **kwargs):
    kwargs["column_format"] = "l|" + "c" * self.shape[1]
    return _orig_to_latex(self, *args, **kwargs)
pd.DataFrame.to_latex = _to_latex

In [19]:
# Lattice von Franziskas MD Simulation (scheinen aus PBE Structure Optimization zu stammen)
lattice_hegner_MD = np.array(
    [
        [12.525945, 0.000000, 0.000000],
        [-6.262972, 10.847786, 0.000000],
        [0.000000, 0.000000, 17.483654],
    ]
)

# crystal angles
alpha_hegner_MD = (
    np.arccos(
        np.dot(lattice_hegner_MD[1], lattice_hegner_MD[2])
        / (np.linalg.norm(lattice_hegner_MD[1]) * np.linalg.norm(lattice_hegner_MD[2]))
    )
    * 180
    / np.pi
)

beta_hegner_MD = (
    np.arccos(
        np.dot(lattice_hegner_MD[0], lattice_hegner_MD[2])
        / (np.linalg.norm(lattice_hegner_MD[0]) * np.linalg.norm(lattice_hegner_MD[2]))
    )
    * 180
    / np.pi
)
gamma_hegner_MD = (
    np.arccos(
        np.dot(lattice_hegner_MD[0], lattice_hegner_MD[1])
        / (np.linalg.norm(lattice_hegner_MD[0]) * np.linalg.norm(lattice_hegner_MD[1]))
    )
    * 180
    / np.pi
)

# lattice aus eigener, mit PBE optimierten Struktur (wurde für MACE MD Simulation verwendet)
lattice_own_MD = np.array(
    [
        [12.613593, 0.0, 0.0],
        [-6.306796, 10.923692, 0.0],
        [0.0, 0.0, 17.544581],
    ]
)

# crystal angles
alpha_own_MD = (
    np.arccos(
        np.dot(lattice_own_MD[1], lattice_own_MD[2])
        / (np.linalg.norm(lattice_own_MD[1]) * np.linalg.norm(lattice_own_MD[2]))
    )
    * 180
    / np.pi
)
beta_own_MD = (
    np.arccos(
        np.dot(lattice_own_MD[0], lattice_own_MD[2])
        / (np.linalg.norm(lattice_own_MD[0]) * np.linalg.norm(lattice_own_MD[2]))
    )
    * 180
    / np.pi
)
gamma_own_MD = (
    np.arccos(
        np.dot(lattice_own_MD[0], lattice_own_MD[1])
        / (np.linalg.norm(lattice_own_MD[0]) * np.linalg.norm(lattice_own_MD[1]))
    )
    * 180
    / np.pi
)

# get lattice parameters and angles from "TaCuN2_unrelaxed.cif" file
with open("TaCuN2_unrelaxed.cif", "r") as f:
    lines = f.readlines()
    for line in lines:
        if line.startswith("_cell_length_a"):
            a_cif = float(line.split()[1])
        elif line.startswith("_cell_length_b"):
            b_cif = float(line.split()[1])
        elif line.startswith("_cell_length_c"):
            c_cif = float(line.split()[1])
        elif line.startswith("_cell_angle_alpha"):
            alpha_cif = float(line.split()[1])
        elif line.startswith("_cell_angle_beta"):
            beta_cif = float(line.split()[1])
        elif line.startswith("_cell_angle_gamma"):
            gamma_cif = float(line.split()[1])


# calculate lattice parameters from lattice vectors
a_hegner_MD = np.linalg.norm(lattice_hegner_MD[0]) / 4
b_hegner_MD = np.linalg.norm(lattice_hegner_MD[1]) / 4
c_hegner_MD = np.linalg.norm(lattice_hegner_MD[2])

# lattice parameters from own MD simulation (unit cell)
a_own_MD = np.linalg.norm(lattice_own_MD[0]) / 4
b_own_MD = np.linalg.norm(lattice_own_MD[1]) / 4
c_own_MD = np.linalg.norm(lattice_own_MD[2])

# Lattice parameters from Yang (unit cell)
a_yang = b_yang = 3.1445
c_yang = 17.413
alpha_yang = beta_yang = 90
gamma_yang = 120


# parse the lattice parameters from the CONTCAR in the relaxation directory
with open("relaxation/CONTCAR", "r") as f:
    lines = f.readlines()
    lattice_own_relaxation = np.array(
        [
            [float(x) for x in lines[2].split()],
            [float(x) for x in lines[3].split()],
            [float(x) for x in lines[4].split()],
        ]
    )
    # find the angles
    alpha_own_relaxation = (
        np.arccos(
            np.dot(lattice_own_relaxation[1], lattice_own_relaxation[2])
            / (
                np.linalg.norm(lattice_own_relaxation[1])
                * np.linalg.norm(lattice_own_relaxation[2])
            )
        )
        * 180
        / np.pi
    )
    beta_own_relaxation = (
        np.arccos(
            np.dot(lattice_own_relaxation[0], lattice_own_relaxation[2])
            / (
                np.linalg.norm(lattice_own_relaxation[0])
                * np.linalg.norm(lattice_own_relaxation[2])
            )
        )
        * 180
        / np.pi
    )
    gamma_own_relaxation = (
        np.arccos(
            np.dot(lattice_own_relaxation[0], lattice_own_relaxation[1])
            / (
                np.linalg.norm(lattice_own_relaxation[0])
                * np.linalg.norm(lattice_own_relaxation[1])
            )
        )
        * 180
        / np.pi
    )

# create a pandas table summarizing the lattice parameters from the different sources
# 6 rows, with a b c alpha beta gamma
# N columns, one for each source (Hegner, Own, Yang, Materials Project)
data = {
    r"Hegner et al. \cite{hegnerCriticalRoleAnharmonic2024}": [
        a_hegner_MD,
        b_hegner_MD,
        c_hegner_MD,
        # alpha_hegner_MD,
        # beta_hegner_MD,
        # gamma_hegner_MD,
    ],
    r"Yang et al. \cite{yangStrongOpticalAbsorption2013}": [
        a_yang,
        b_yang,
        c_yang,
        # alpha_yang,
        # beta_yang,
        # gamma_yang,
    ],
    r"MP \cite{noneavailableMaterialsDataTaCuN22017}": [
        a_cif,
        b_cif,
        c_cif,
        # alpha_cif,
        # beta_cif,
        # gamma_cif,
    ],
    "Our relaxed structure": [
        np.linalg.norm(lattice_own_relaxation[0]),
        np.linalg.norm(lattice_own_relaxation[1]),
        np.linalg.norm(lattice_own_relaxation[2]),
        # alpha_own_relaxation,
        # beta_own_relaxation,
        # gamma_own_relaxation,
    ],
}
df = pd.DataFrame(data)

# change the row labels to source, a, b, c, alpha, beta, gamma
df.index = [r"a (\unit{\angstrom})", r"b (\unit{\angstrom})", r"c (\unit{\angstrom})"]

# export the table to a LaTeX table in the output directory
# make it so that the columns are labeled as "Authors"
df.columns.name = "Authors"

# compute cell volumes
# reduce hegner cell (it is a 4x4x1 supercell) to a 1x1x1 cell
hegner_cell_reduced = np.array(
    [
        [
            lattice_hegner_MD[0][0] / 4,
            lattice_hegner_MD[0][1] / 4,
            lattice_hegner_MD[0][2] / 4,
        ],
        [
            lattice_hegner_MD[1][0] / 4,
            lattice_hegner_MD[1][1] / 4,
            lattice_hegner_MD[1][2] / 4,
        ],
        [lattice_hegner_MD[2][0], lattice_hegner_MD[2][1], lattice_hegner_MD[2][2]],
    ]
)


vol_hegner = 148.44
vol_yang = 149.09
vol_cif = (
    a_cif
    * b_cif
    * c_cif
    * np.sqrt(
        1
        - np.cos(np.radians(alpha_cif)) ** 2
        - np.cos(np.radians(beta_cif)) ** 2
        - np.cos(np.radians(gamma_cif)) ** 2
        + 2
        * np.cos(np.radians(alpha_cif))
        * np.cos(np.radians(beta_cif))
        * np.cos(np.radians(gamma_cif))
    )
)
vol_own = np.linalg.det(lattice_own_relaxation)

# add these as a new row to the table
df.loc[r"Volume ($\unit{\angstrom^3}$)"] = [vol_hegner, vol_yang, vol_cif, vol_own]

df.to_latex(
    "output/lattice_parameters.tex",
    caption="Lattice parameters from different sources.",
    label="tab:lattice_parameters",
    float_format=lambda x: f"{x:.4g}",
)
df

Authors,Hegner et al. \cite{hegnerCriticalRoleAnharmonic2024},Yang et al. \cite{yangStrongOpticalAbsorption2013},MP \cite{noneavailableMaterialsDataTaCuN22017},Our relaxed structure
a (\unit{\angstrom}),3.131486,3.1445,3.153398,3.151112
b (\unit{\angstrom}),3.131486,3.1445,3.153398,3.151112
c (\unit{\angstrom}),17.483654,17.4130,17.544581,17.572585
Volume ($\unit{\angstrom^3}$),148.440000,149.0900,151.088447,151.110267


In [ ]:
# open relaxation/POSCAR

with open("relaxation/POSCAR", "r") as f:
    poscar_lines = f.readlines()

# extract lattice vectors from POSCAR
lattice_poscar = np.array(
    [
        [float(x) for x in poscar_lines[2].split()],
        [float(x) for x in poscar_lines[3].split()],
        [float(x) for x in poscar_lines[4].split()],
    ]
)

# 4x4x1 supercell
lattice_poscar_supercell = lattice_poscar * np.array([[4, 0, 0], [0, 4, 0], [0, 0, 1]])
lattice_poscar_supercell

array([[12.6135928 ,  0.        ,  0.        ],
       [-0.        , 10.9236918 ,  0.        ],
       [ 0.        ,  0.        , 17.54458073]])

In [ ]:
# find cell volumes
volume_hegner = np.abs(np.linalg.det(lattice_hegner))
volume_own = np.abs(np.linalg.det(lattice_own))

print(f"Cell volume from Hegner: {volume_hegner:.6f}")
print(f"Cell volume from Own: {volume_own:.6f}")

# calculate how much percent the cell volumes differ
volume_diff = (volume_own - volume_hegner) / volume_hegner * 100
print(f"Cell volume difference: {volume_diff:.6f} %")

NameError: name 'lattice_hegner' is not defined